# Feature Engineering

## Purpose

This notebook transforms the cleaned 2025 Airline On-Time Performance dataset into model-ready features for schedule-time flight-delay prediction.

Only information available before the scheduled departure of a flight will be used as predictive input. Variables generated during or after flight operations are excluded to prevent target leakage.

The feature-engineering workflow includes:

1. Loading and validating the cleaned Delta table
2. Creating the model-eligible flight population
3. Engineering temporal features
4. Engineering schedule features
5. Selecting the final predictive variables
6. Validating the engineered dataset
7. Saving the feature dataset for model training

## 1. Load and Validate the Cleaned Dataset

The feature-engineering process begins by loading the managed `flights_clean` Delta table produced by the data-cleaning notebook.

The source table is validated before transformations are applied. The raw and cleaned datasets remain unchanged throughout this notebook.

In [0]:
from __future__ import annotations

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window


CLEAN_TABLE = "workspace.default.flights_clean"
FEATURE_TABLE = "workspace.default.flights_features"
TARGET_COLUMN = "ARR_DEL15"


def require_table(table_name: str) -> None:
    """Raise an error when a required Unity Catalog table is unavailable."""
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run the data-cleaning notebook before continuing."
        )


require_table(CLEAN_TABLE)

df_clean: DataFrame = spark.table(CLEAN_TABLE)

clean_row_count = df_clean.count()
clean_column_count = len(df_clean.columns)

required_columns = {
    "FL_DATE",
    "QUARTER",
    "MONTH",
    "DAY_OF_WEEK",
    "OP_UNIQUE_CARRIER",
    "OP_CARRIER_FL_NUM",
    "ORIGIN",
    "DEST",
    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    "CRS_ELAPSED_TIME",
    "DISTANCE",
    "ARR_DEL15",
    "CANCELLED",
    "DIVERTED",
}

missing_columns = sorted(required_columns - set(df_clean.columns))

if missing_columns:
    raise ValueError(
        "Feature-engineering validation failed. "
        f"Missing required columns: {missing_columns}"
    )

print("Cleaned dataset loaded and validated successfully.")
print(f"Source table: {CLEAN_TABLE}")
print(f"Output table: {FEATURE_TABLE}")
print(f"Total records: {clean_row_count:,}")
print(f"Total columns: {clean_column_count}")
print(f"Prediction target: {TARGET_COLUMN}")

## 2. Create the Model-Eligible Dataset

The cleaned dataset contains all historical flight records, including cancelled and diverted flights.

For supervised machine learning, only flights that completed normal arrival operations are retained. Cancelled and diverted flights are excluded because the prediction target (`ARR_DEL15`) represents arrival delay for completed flights.

This filtering establishes the final population used for feature engineering and model development while preserving the original cleaned dataset for descriptive and diagnostic analyses.

In [0]:
df_model = (
    df_clean
    .filter(
        (F.col("CANCELLED") == 0)
        &
        (F.col("DIVERTED") == 0)
    )
)

model_row_count = df_model.count()

removed_records = clean_row_count - model_row_count

model_summary = spark.createDataFrame(
    [
        (
            clean_row_count,
            model_row_count,
            removed_records,
        )
    ],
    schema="""
        CLEAN_DATASET_ROWS long,
        MODEL_DATASET_ROWS long,
        EXCLUDED_RECORDS long
    """
)

display(model_summary)

### Validation Summary

The model-eligible dataset contains only completed flights that are suitable for supervised learning.

Cancelled and diverted flights were excluded because they do not represent standard arrival operations and therefore cannot be used to train a model that predicts arrival delay.

The filtered dataset will serve as the basis for all subsequent feature engineering activities.

## 3. Engineer Temporal Features

Temporal features capture calendar- and time-related patterns that may influence flight delays. These variables are derived exclusively from information available before the scheduled departure of a flight and therefore comply with the project's target leakage restrictions.

The following temporal features are created:

- `DEP_HOUR` – Scheduled departure hour extracted from `CRS_DEP_TIME`
- `DEP_MINUTE` – Scheduled departure minute extracted from `CRS_DEP_TIME`
- `IS_WEEKEND` – Indicates whether the scheduled flight departs on a weekend
- `SEASON` – Meteorological season derived from the flight month

These engineered features improve the model's ability to learn recurring temporal patterns in airline operations.

In [0]:
df_features = df_model

# Scheduled departure hour
df_features = df_features.withColumn(
    "DEP_HOUR",
    F.floor(F.col("CRS_DEP_TIME") / 100).cast("int"),
)

# Scheduled departure minute
df_features = df_features.withColumn(
    "DEP_MINUTE",
    (F.col("CRS_DEP_TIME") % 100).cast("int"),
)

# Weekend indicator:
# 1 = Monday, 6 = Saturday, 7 = Sunday
df_features = df_features.withColumn(
    "IS_WEEKEND",
    F.when(
        F.col("DAY_OF_WEEK").isin(6, 7),
        F.lit(1),
    ).otherwise(F.lit(0)),
)

# Meteorological season
df_features = df_features.withColumn(
    "SEASON",
    F.when(F.col("MONTH").isin(12, 1, 2), "Winter")
    .when(F.col("MONTH").isin(3, 4, 5), "Spring")
    .when(F.col("MONTH").isin(6, 7, 8), "Summer")
    .otherwise("Fall"),
)

print("Temporal features created successfully.")
print(f"Rows retained: {df_features.count():,}")
print(f"Columns after temporal engineering: {len(df_features.columns)}")

### Validation of Engineered Temporal Features

The newly engineered temporal features are displayed below to verify that they were generated correctly.

The validation confirms that:

- `DEP_HOUR` correctly represents the scheduled departure hour.
- `DEP_MINUTE` correctly represents the scheduled departure minute.
- `IS_WEEKEND` follows the BTS day-of-week convention.
- `SEASON` has been correctly derived from the scheduled flight month.

These features will be used in the predictive model because they are known before the scheduled departure of a flight.

In [0]:
display(

    df_features.select(

        "FL_DATE",
        "MONTH",
        "DAY_OF_WEEK",
        "CRS_DEP_TIME",

        "DEP_HOUR",
        "DEP_MINUTE",
        "IS_WEEKEND",
        "SEASON"

    ).limit(20)

)

### Validation Summary

The distribution of the engineered temporal features is examined to verify that all expected categories have been generated correctly across the complete model population.

The following summaries confirm the creation of:

- Meteorological seasons
- Weekend indicators
- Scheduled departure hours

In [0]:
display(

    df_features
    .groupBy("SEASON")
    .count()
    .orderBy("SEASON")

)

display(

    df_features
    .groupBy("IS_WEEKEND")
    .count()
    .orderBy("IS_WEEKEND")

)

display(

    df_features
    .groupBy("DEP_HOUR")
    .count()
    .orderBy("DEP_HOUR")

)

## 4. Engineer Schedule Features

Schedule-related features describe operational characteristics that are known before departure. These features complement the temporal variables created previously and provide additional information about the scheduled flight.

The following schedule features are engineered:

- `TIME_OF_DAY` – Categorizes scheduled departures into operational periods.
- `FLIGHT_DISTANCE_CATEGORY` – Groups flights into short-, medium-, and long-haul categories based on scheduled distance.

These features are derived exclusively from scheduled flight information and therefore satisfy the project's target leakage restrictions.

In [0]:
# -------------------------------------------------------
# Time of Day
# -------------------------------------------------------

df_features = df_features.withColumn(
    "TIME_OF_DAY",
    F.when(F.col("DEP_HOUR").between(0, 5), "Overnight")
     .when(F.col("DEP_HOUR").between(6, 11), "Morning")
     .when(F.col("DEP_HOUR").between(12, 16), "Afternoon")
     .when(F.col("DEP_HOUR").between(17, 20), "Evening")
     .otherwise("Night")
)

# -------------------------------------------------------
# Flight Distance Category
# -------------------------------------------------------

df_features = df_features.withColumn(
    "FLIGHT_DISTANCE_CATEGORY",
    F.when(F.col("DISTANCE") < 500, "Short")
     .when(F.col("DISTANCE") <= 1500, "Medium")
     .otherwise("Long")
)

print("Schedule features created successfully.")
print(f"Current columns: {len(df_features.columns)}")

### Validation of Schedule Features

The engineered schedule features are summarized below to verify that the new categories have been assigned correctly.

The validation confirms:

- Scheduled departure times were categorized into the appropriate operational periods.
- Flight distances were grouped into short-, medium-, and long-haul categories.

In [0]:
display(
    df_features.groupBy("TIME_OF_DAY")
    .count()
    .orderBy("TIME_OF_DAY")
)

display(
    df_features.groupBy("FLIGHT_DISTANCE_CATEGORY")
    .count()
    .orderBy("FLIGHT_DISTANCE_CATEGORY")
)

## 5. Select Final Predictive Variables

The engineered dataset still contains operational variables that are useful for descriptive and diagnostic analytics but cannot be used for schedule-time prediction.

Following the project's target leakage restrictions, only variables available before the scheduled departure of a flight are retained for machine learning.

The final predictive dataset therefore consists of:

- Calendar variables
- Airline information
- Route information
- Geographic information
- Scheduled flight information
- Engineered temporal features
- Engineered schedule features
- Target variable (`ARR_DEL15`)

All variables generated during or after flight operations are excluded from the predictive dataset.

In [0]:
FINAL_FEATURE_COLUMNS = [

    # -----------------------------
    # Calendar
    # -----------------------------
    "QUARTER",
    "MONTH",
    "DAY_OF_WEEK",
    "FL_DATE",

    # -----------------------------
    # Airline
    # -----------------------------
    "OP_UNIQUE_CARRIER",

    # -----------------------------
    # Route
    # -----------------------------
    "ORIGIN",
    "DEST",
    "DISTANCE",

    # -----------------------------
    # Geographic
    # -----------------------------
    "ORIGIN_CITY_NAME",
    "ORIGIN_STATE_NM",
    "DEST_CITY_NAME",
    "DEST_STATE_NM",

    # -----------------------------
    # Schedule
    # -----------------------------
    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    "CRS_ELAPSED_TIME",

    # -----------------------------
    # Engineered Temporal Features
    # -----------------------------
    "DEP_HOUR",
    "DEP_MINUTE",
    "IS_WEEKEND",
    "SEASON",

    # -----------------------------
    # Engineered Schedule Features
    # -----------------------------
    "TIME_OF_DAY",
    "FLIGHT_DISTANCE_CATEGORY",

    # -----------------------------
    # Target
    # -----------------------------
    "ARR_DEL15",
]

df_ml = df_features.select(*FINAL_FEATURE_COLUMNS)

print("Final predictive dataset created successfully.")
print(f"Rows: {df_ml.count():,}")
print(f"Columns: {len(df_ml.columns)}")

### Validation of Selected Predictive Variables

The final predictive dataset is verified to ensure that only approved schedule-time variables are retained.

This validation confirms that:

- All leakage variables have been excluded.
- All engineered predictive features have been retained.
- The target variable (`ARR_DEL15`) is present.
- The dataset is ready for model training.

In [0]:
print("Final Predictive Variables")

for column in df_ml.columns:
    print(column)

## 6. Validate the Engineered Dataset

The engineered dataset is validated before being used for machine learning model development.

The validation confirms:

- Total number of records
- Total number of predictive variables
- Missing values in each selected feature
- Data types of engineered variables

This final validation ensures that the predictive dataset satisfies the project's feature engineering and target leakage requirements.

In [0]:
validation_summary = spark.createDataFrame(
    [
        (
            df_ml.count(),
            len(df_ml.columns),
        )
    ],
    schema="""
        TOTAL_RECORDS long,
        TOTAL_FEATURES long
    """
)

display(validation_summary)

### Validation of Missing Values

The final predictive dataset is examined to identify any remaining missing values within the selected machine learning features.

Because all leakage variables have been removed and incomplete records were addressed during the data-cleaning stage, no missing values are expected within the retained predictive variables.

This validation confirms the completeness of the final feature dataset before model training.

In [0]:
null_summary = []

total_rows = df_ml.count()

for column_name in df_ml.columns:

    null_count = (
        df_ml
        .filter(F.col(column_name).isNull())
        .count()
    )

    null_percentage = round(
        (null_count / total_rows) * 100,
        4
    )

    null_summary.append(
        (
            column_name,
            null_count,
            null_percentage,
        )
    )

null_summary_df = spark.createDataFrame(
    null_summary,
    [
        "COLUMN",
        "NULL_COUNT",
        "NULL_PERCENTAGE",
    ],
)

display(
    null_summary_df.orderBy(
        F.col("NULL_COUNT").desc()
    )
)

### Validation of Data Types

The schema of the engineered dataset is reviewed to verify that all predictive variables have appropriate data types before model training.

Numerical, categorical, temporal, and engineered variables should each retain consistent data types suitable for subsequent encoding and machine learning.

In [0]:
schema_rows = []

for field in df_ml.schema.fields:

    schema_rows.append(

        (
            field.name,
            field.dataType.simpleString(),
            field.nullable,
        )

    )

schema_df = spark.createDataFrame(

    schema_rows,

    [
        "COLUMN_NAME",
        "DATA_TYPE",
        "NULLABLE",
    ]

)

display(schema_df)

## 7. Save the Feature Dataset

The validated feature dataset is stored as a managed Delta table within Unity Catalog.

This dataset contains only the approved schedule-time predictive variables together with the engineered temporal and schedule features required for machine learning.

The resulting table will be used as the input dataset for the Model Training notebook.

In [0]:
FEATURE_TABLE = "workspace.default.flights_features"

(
    df_ml.writeTo(FEATURE_TABLE)
    .using("delta")
    .createOrReplace()
)

print("Feature dataset saved successfully.")
print(f"Table: {FEATURE_TABLE}")
print(f"Rows: {spark.table(FEATURE_TABLE).count():,}")

### Validation Summary

The managed Delta table is reloaded from Unity Catalog to verify that the feature dataset has been written successfully.

The verification confirms that:

- The table is accessible.
- The record count matches the engineered dataset.
- The schema has been preserved.

The dataset is now ready for machine learning model development.

### Store a Copy in the Processed Layer

In addition to registering the engineered dataset as a managed Unity Catalog table, a Delta copy is stored in the project's `processed` layer.

Maintaining both the managed table and the processed Delta files provides a clear data lake organization while allowing downstream notebooks to access the data through Unity Catalog.

In [0]:
FEATURE_DELTA_PATH = (
    "/Volumes/workspace/default/"
    "flight_delay_capstone/processed/flights_features"
)

(
    df_ml.write
    .format("delta")
    .mode("overwrite")
    .save(FEATURE_DELTA_PATH)
)

print("Feature dataset copied to processed layer.")
print(f"Location: {FEATURE_DELTA_PATH}")

In [0]:
df_feature_copy = spark.read.format("delta").load(FEATURE_DELTA_PATH)

print(f"Rows: {df_feature_copy.count():,}")
print(f"Columns: {len(df_feature_copy.columns)}")

display(df_feature_copy.limit(10))